# Cleaning and Filtering the Movies Dataset

In the first notebook, we conducted an Exploratory Data Analysis (EDA) on the movies dataset to ensure we have enough data and understand its structure. In this notebook, I will clean and filter some movies so we have only relevant films for our recommender system

In [88]:
import pandas as pd
import ast
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Movies Dataset

In [89]:
mdf = pd.read_parquet('../Data/Raw/movies_metadata.parquet')
mdf.columns

Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'origin_country', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'video', 'vote_average', 'vote_count', 'movieId'],
      dtype='object')

In [90]:
# Select only the relevant columns
mdf = mdf[['movieId', 'id', 'title', 'genres', 'overview', 
           'release_date', 'runtime', 'tagline',  'vote_average', 
           'popularity', 'vote_count', 'poster_path', 'backdrop_path']]

mdf.head().transpose()

,0,1,2,3,4
movieId,16,11,7,8,1
id,524,9087,11860,45325,862
title,Casino,The American President,Sabrina,Tom and Huck,Toy Story
genres,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[{'id': 10751, 'name': 'Family'}, {'id': 28, '...","[{'id': 16, 'name': 'Animation'}, {'id': 12, '..."
overview,"In early-1970s Las Vegas, Sam ""Ace"" Rothstein ...","Widowed U.S. president Andrew Shepherd, one of...","Sabrina Fairchild, a chauffeur's daughter, gre...","A mischievous young boy, Tom Sawyer, witnesses...","Led by Woody, Andy's toys live happily in his ..."
release_date,1995-11-22,1995-11-17,1995-12-15,1995-12-22,1995-11-22
runtime,179,114,127,97,81
tagline,No one stays at the top forever.,Why can't the most powerful man in the world h...,You are cordially invited to the most surprisi...,A lot of kids get into trouble. These two inve...,Hang on for the comedy that goes to infinity a...
vote_average,7.997,6.5,6.204,5.3,7.968
popularity,6.6361,1.6287,2.7156,0.7317,22.3485


In [91]:
mdf.dtypes

movieId            int64
id                 int64
title             object
genres            object
overview          object
release_date      object
runtime            int64
tagline           object
vote_average     float64
popularity       float64
vote_count         int64
poster_path       object
backdrop_path     object
dtype: object

It seems that all the features are in its correct data type

In [92]:
print(f'The original movies dataset has {mdf.shape[0]:,} movies')

The original movies dataset has 86,351 movies


## Cleaning and Preprocessing  the Dataset

We will clean the dataset by removing rows that lack important information. Specifically, we will remove movies that do not have a `title`, `overview`, or `genre`, as these are essential for identifying movies and calculating similarities in a content-based recommender system. Without a title, we cannot determine which movie it is, and without an overview or genre, we cannot derive meaningful movie comparisons.

Also we will transform the data types of certain columns to ensure they are properly formatted for further analysis. Specifically, we will extract and convert the genres, collection (if available), and production companies, as these columns are currently stored as stringified dictionaries.


In [93]:
# Drop films without title or genre
mdf.dropna(subset='title', inplace=True)
mdf.dropna(subset='overview', inplace=True)

In [94]:
def extract_dict(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return np.nan

In [95]:
# Extract the genres
mdf['genres'] =  mdf['genres'].apply(lambda x: ast.literal_eval(x)).apply(lambda x: [item['name'] for item in x])
mdf.head().transpose()

,0,1,2,3,4
movieId,16,11,7,8,1
id,524,9087,11860,45325,862
title,Casino,The American President,Sabrina,Tom and Huck,Toy Story
genres,"[Crime, Drama]","[Comedy, Drama, Romance]","[Romance, Drama]","[Family, Action, Adventure, Drama]","[Animation, Adventure, Family, Comedy]"
overview,"In early-1970s Las Vegas, Sam ""Ace"" Rothstein ...","Widowed U.S. president Andrew Shepherd, one of...","Sabrina Fairchild, a chauffeur's daughter, gre...","A mischievous young boy, Tom Sawyer, witnesses...","Led by Woody, Andy's toys live happily in his ..."
release_date,1995-11-22,1995-11-17,1995-12-15,1995-12-22,1995-11-22
runtime,179,114,127,97,81
tagline,No one stays at the top forever.,Why can't the most powerful man in the world h...,You are cordially invited to the most surprisi...,A lot of kids get into trouble. These two inve...,Hang on for the comedy that goes to infinity a...
vote_average,7.997,6.5,6.204,5.3,7.968
popularity,6.6361,1.6287,2.7156,0.7317,22.3485


Delete the films of which we do not have their genres

In [96]:
mdf['genres'] = mdf['genres'].apply(lambda x: np.nan if not x else x)
mdf.dropna(subset='genres', inplace=True)

#### Poster Path

The poster path will be useful to display the poster of the movie in the user interface, as well as the backdrop poster; so let's check how many posters are missing

In [97]:
mdf['poster_path'].isnull().sum()

np.int64(1111)

In [98]:
mdf['backdrop_path'].isnull().sum()

np.int64(10890)

In [99]:
mdf[mdf['poster_path'].isnull()].sort_values(by='popularity', ascending=False)[['title', 'popularity']].head()

,title,popularity
32843,Carlos Spills the Beans,1.7193
22353,Hard Sun,1.5490
22395,Serial Killer Culture,1.2484
31976,Targeting,1.1432
44121,Trailer Park Boys: Swearnet Live,1.0292


The movies without a poster path appear to be quite unpopular. Since the posters are essential for displaying the film in the final UI, we will remove the films that lack a poster path. Also notice that the films that lack of this information are not populars or have a lot of votes.

In [100]:
mdf[mdf['backdrop_path'].isnull()].\
    sort_values(by='popularity', ascending=False)[['title', 'popularity', 'vote_average', 'vote_count']].head()

,title,popularity,vote_average,vote_count
16512,Girl Play,12.0767,5.447,19
29675,Snow White and 7 Wise Men,9.1104,5.000,13
46479,Phenomenon II,6.2001,6.000,11
60225,The Last Best Sunday,3.3010,5.000,4
31918,The Party at Kitty and Stud's,2.8393,3.800,72


In [101]:
mdf.dropna(subset='poster_path', inplace=True)

In [102]:
mdf.dropna(subset='backdrop_path', inplace=True)

In [103]:
print(f'After doing some cleaning we are left with {mdf.shape[0]:,} movies')

After doing some cleaning we are left with 73,809 movies


## Filtering

To enhance the performance of our recommender system, we will filter and select only relevant movies for the following reasons:

- **Relevance to Modern Audiences:** Older movies may not resonate with today’s viewers. By focusing on more recent or popular titles, we ensure that recommendations remain aligned with current trends and user preferences.
- **Avoiding Data Sparsity:** Older movies typically have fewer interactions and ratings, leading to data sparsity. Since recommender systems rely on user interactions, movies with limited data may not generate meaningful recommendations.
- **Reducing Complexity:** A smaller, more focused dataset of relevant movies reduces model complexity. Working with fewer, more relevant movies means fewer features to process, resulting in faster computation and more efficient learning.
- **Improving User-Item Interactions:** Users are generally more engaged with recent or trending movies. Filtering out older titles helps focus the system on movies with more active user interactions, thereby enhancing the model’s accuracy.

Filtering movies by vote count, popularity, or average rating can often lead to misleading results. To address this, we apply IMDB's weighted rating formula, which balances a movie's average rating with the number of votes it has received. In other words, a high rating alone isn't enough — the movie also needs a significant number of votes to be considered trustworthy.

$$\text{WR} = \left( \frac{v}{v + m} \cdot R \right) + \left( \frac{m}{v + m} \cdot C \right)$$

where:

  - $v$ is the number of votes for the movie
  - $m$ is the minimum number of votes required for relevance
  - $R$ is the average rating of the movie
  - $C$ is the mean rating across all movies in the dataset

Since we have nearly 90,000 movies and 75% of them have 82 votes or fewer (which is relatively low), we will set $m$ to the 90th percentile of the vote count distribution. This means a movie must have more votes than 90\% of all movies in the dataset to be considered relevant.

In [104]:
m = mdf['vote_count'].quantile(0.9)
C = mdf['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

In [105]:
mdf['score'] = mdf.apply(weighted_rating, axis=1)

# Top 5 movies according to IMDb's score
mdf.sort_values(by='score', ascending=False).head()

,movieId,id,title,genres,overview,release_date,runtime,tagline,vote_average,popularity,vote_count,poster_path,backdrop_path,score
314,318,278,The Shawshank Redemption,"[Drama, Crime]",Imprisoned in the 1940s for the double murder ...,1994-09-23,142,Fear can hold you prisoner. Hope can set you f...,8.700,38.2989,28003,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,8.656773
840,858,238,The Godfather,"[Drama, Crime]","Spanning the years 1945 to 1955, a chronicle o...",1972-03-14,175,An offer you can't refuse.,8.689,36.5907,21223,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,/tmU7GeKVybMWFButWEGl2M4GeiP.jpg,8.632483
521,527,424,Schindler's List,"[Drama, History, War]",The true story of how businessman Oskar Schind...,1993-12-15,195,"Whoever saves one life, saves the world entire.",8.566,20.2365,16289,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,8.496163
12170,58559,155,The Dark Knight,"[Drama, Action, Crime, Thriller]",Batman raises the stakes in his war on crime. ...,2008-07-16,152,Welcome to a world without rules.,8.519,29.5713,33625,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/oOv2oUXcAaNXakRqUPxYq5lJURz.jpg,8.485315
1184,1221,240,The Godfather Part II,"[Drama, Crime]",In the continuing saga of the Corleone crime f...,1974-12-20,202,The rise and fall of the Corleone empire.,8.569,17.7084,12822,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,8.480823


As we can see this filtering step was pretty good, since the top movies we can see are authentic gems.

### Year & Rating

We will extract the release year of the films and select movies released in 1990 or later, since our target audience is mostly yougn people, born in the 2000's.

In [106]:
mdf['release_date'] = pd.to_datetime(mdf['release_date'], errors='coerce')
mdf['year'] = mdf['release_date'].dt.year.fillna(1989).astype('int')
# Drop the release date
mdf = mdf.drop(columns=['release_date']).reset_index(drop=True)
# Filter the movies
mdf = mdf[ (mdf['year'] > 1994)]

Finally we will select the best 5,000 movies according to this score

In [107]:
mdf = mdf.sort_values(by='score', ascending=False).head(5000)

# Ensure the size is the correct
mdf.shape

(5000, 14)

In [ ]:
# Save only the ids on a csv file 
mdf[['movieId', 'id']].to_csv('../Data/Processed/movies_ids.csv', index=False)